# Exploring the course datasets

Welcome! In this notebook you will meet the two datasets that we will use throughout the course, and along the way refresh the basic data-handling skills that the course builds on.

By the end you will have:

1. Loaded both course datasets (**20 Newsgroups** and **TweetEval**) and learned where they come from;
2. Practised the core `pandas` operations for inspecting and manipulating tabular data: viewing, counting, filtering, string operations, grouping, and simple plots;
3. Seen, hands-on, what "messy" text data looks like -- and why the *length* and *style* of documents matters for the methods you will learn.

Work through it top to bottom, running each cell (`Shift + Enter`). Expect it to take roughly 60-90 minutes. Small **Your turn** exercises are scattered throughout; solutions are at the very bottom, but try first!

> If any cell errors in a way you cannot resolve, make a note and bring it to Day 1 -- that is exactly what the morning setup check is for.

## 0. Setup

We start by importing the libraries we need. Running this cell first also serves as a check that your course environment is correctly installed -- if everything was set up according to the setup guide, it should complete without errors.

In [ ]:
import pandas as pd                              #dataframes: tabular data with named columns
import matplotlib.pyplot as plt                  #plotting
from sklearn.datasets import fetch_20newsgroups  #downloads the 20 Newsgroups dataset for us
from datasets import load_dataset                #downloads datasets from the Hugging Face Hub

print("All imports succeeded - your environment is ready.")

## 1. Dataset one: 20 Newsgroups

**What it is.** Around 18,000 messages posted in 1994-1995 to *Usenet newsgroups* -- public discussion forums that predate the web forums and social media platforms of today. Each message belongs to one of 20 groups, each dedicated to a topic: ice hockey, cryptography, space, Middle East politics, computer hardware, and so on.

**Why we use it.** It is one of the classic datasets of text analysis: built into scikit-learn (no download hassle), a manageable size, and -- most importantly -- it has a *known* topical structure. When we cluster it or run topic models over it on Day 2, we can check whether the structure our methods discover matches the structure we know is there. That makes it ideal for learning.

**Its limitation.** The data shows its age: 1990s American internet culture, English only, email-style formatting. Keep that in mind -- it is a teaching corpus, not a template for your own research data.

More information: the [scikit-learn documentation](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset) and the [original dataset homepage](http://qwone.com/~jason/20Newsgroups/).

Let's fetch it (the first run downloads ~14 MB; afterwards it is cached on your machine):

In [ ]:
newsgroups = fetch_20newsgroups(subset="all")  #subset='all' gives us every message

#the result is a container with the texts, the numeric labels, and the label names:
print(type(newsgroups))
print(f"number of documents: {len(newsgroups.data)}")
print(f"number of categories: {len(newsgroups.target_names)}")
print(newsgroups.target_names)

### What does one document look like?

Before doing anything clever with text data, **read some of it**. This is the single most underrated step in computational text analysis. Let's print one raw message:

In [ ]:
print(newsgroups.data[0])

Notice how much of that is *not* content: email headers (`From:`, `Subject:`, `Organization:`), quoted text from earlier messages, signatures. This is what "messy text" means in practice, and every real-world text collection has its own version of it (think of retweets, HTML tags, or boilerplate in news articles).

Left in place, this boilerplate can badly mislead our models: a classifier might "learn" to recognise the *hockey* group by the email addresses of its frequent posters rather than by anything about hockey. scikit-learn therefore offers to strip it for us -- and this cleaned version is what we will use during the course:

In [ ]:
newsgroups = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"))

#the same document, after cleaning:
print(newsgroups.data[0])

### Into a DataFrame

For exploration, it is convenient to put the texts and their category labels side by side in a pandas **DataFrame** -- a table with named columns, the workhorse of data analysis in Python. The numeric labels (0-19) are not very readable, so we also translate them into the category *names*:

In [ ]:
df = pd.DataFrame({
    "text": newsgroups.data,
    "label": newsgroups.target,
})

#translate numeric labels into readable category names:
#(newsgroups.target_names is a list, so label 0 -> first name, label 1 -> second name, etc.)
df["category"] = df["label"].map(lambda i: newsgroups.target_names[i])

df.head()  #.head() shows the first five rows - always your first look at any DataFrame

In [ ]:
df.info()  #.info() summarises the table: how many rows, which columns, their types

### How are the categories distributed?

`value_counts()` counts how often each unique value occurs in a column. It is the fastest way to understand a categorical variable:

In [ ]:
df["category"].value_counts()

The categories are roughly *balanced* -- each has around 700-1,000 documents. Remember this; the second dataset will look very different.

### Filtering rows

Often you want to look at a subset. In pandas you filter by writing a condition inside square brackets. Let's pull out only the hockey discussions and read one:

In [ ]:
hockey = df[df["category"] == "rec.sport.hockey"]  #keep only rows where the condition is True
print(f"{len(hockey)} hockey documents\n")

#.sample() draws random rows; random_state makes the draw reproducible
print(hockey["text"].sample(1, random_state=42).iloc[0])

### String operations and document length

pandas lets you apply string operations to a whole column at once via the `.str` accessor. A simple but surprisingly informative property of a text collection is how *long* its documents are:

In [ ]:
df["n_chars"] = df["text"].str.len()                    #characters per document
print(df["n_chars"].iloc[0])                            #print a single value from the new column as example: this shows the first text has 712 characters
df["n_words"] = df["text"].str.split().str.len()        #words per document (split on whitespace)
print(df["n_words"].iloc[0])                            #print a single value from the new column as example: this shows the first text has 137 words

df["n_words"].describe()  #.describe() gives summary statistics for a numeric column

The *median* document has under a hundred words, but the maximum is in the tens of thousands -- text length distributions are almost always heavily skewed like this. A histogram shows it at a glance (we cap the x-axis, otherwise the outliers squash everything):

In [ ]:
df["n_words"].plot.hist(bins=100, range=(0, 1000), title="20 Newsgroups: words per document")
plt.xlabel("number of words")
plt.show()

### Grouping

`groupby` splits the data by a categorical column and computes something per group -- the pandas equivalent of a pivot table. Do the discussion topics differ in how long-winded they are?

In [ ]:
df.groupby("category")["n_words"].median().sort_values(ascending=False)

### Your turn (1)

In the empty cell below:

- **a)** Find the single longest document in the dataset (by word count) and print its category. *Hint: `.idxmax()` gives the row position of a column's maximum.*
- **b)** How many documents contain the word `"NASA"`? *Hint: `df["text"].str.contains(...)` gives True/False per row, and `.sum()` counts the Trues.*
- **c)** Which *category* do most of those NASA-mentioning documents belong to? Does the answer make sense?

In [ ]:
#your code here


## 2. Dataset two: TweetEval

**What it is.** [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) (Barbieri et al., 2020) is a benchmark of seven tweet-classification tasks -- offensive language, hate speech, emotion, sentiment, irony, stance, emoji -- each with human-provided labels and fixed train/validation/test splits. We will mainly use the **offensive** task: given a tweet, was it labelled offensive or not?

**Why we use it.** On Days 3-4 you will train models that *predict* these labels -- supervised machine learning. For that we need data where each text comes with a human judgment, and TweetEval provides exactly that, in a clean and well-documented form. It also lets you choose the task that interests you most: the same code will run on any of the tasks by changing a single word.

**Where it lives.** Unlike 20 Newsgroups (bundled with scikit-learn), TweetEval lives on the [Hugging Face Hub](https://huggingface.co/datasets) -- the central repository where the machine learning community shares datasets and models. You will get to know the Hub well during this course. Every dataset there has a *dataset card* describing its contents, origin, and intended use: reading the card before using a dataset is good practice, in the same way you would read the codebook of a survey dataset. Have a look: [TweetEval's dataset card](https://huggingface.co/datasets/cardiffnlp/tweet_eval).

> **A note on content:** this dataset exists to study offensive language online, so a portion of the tweets are exactly that -- offensive, sometimes strongly so. This is normal (and unavoidable) in research on online toxicity, but be prepared for it when reading samples.

Let's load the *offensive* task:

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "offensive")
tweets

### Train, validation, test

Notice that we did not get one table but **three**: `train`, `validation`, and `test`. This split is fundamental to supervised machine learning, and you will work with it constantly from Day 3 on. The short version:

- the model *learns* from the **training** set;
- we *tune our choices* on the **validation** set;
- and we measure final performance on the **test** set, which the model has never seen -- an honest exam, with questions it could not have memorised.

TweetEval fixes these splits for everyone, so results are comparable across research groups (and across course participants!).

Each split is a Hugging Face `Dataset` object. It knows its own structure -- including what the labels mean:

In [ ]:
print(tweets["train"].features)
print(tweets["train"].features["label"].names)  #what do the numeric labels stand for?

So label `0` means *not offensive* and `1` means *offensive*. For exploring, we convert the training split to a pandas DataFrame -- and then everything you practised above applies directly:

In [ ]:
tw = tweets["train"].to_pandas()
tw["label_name"] = tw["label"].map({0: "not-offensive", 1: "offensive"})

print(tw.shape)
tw.head()

### Class imbalance -- a first look at an important problem

How many tweets carry each label?

In [ ]:
print(tw["label_name"].value_counts())
print()
print(tw["label_name"].value_counts(normalize=True).round(3))  #as proportions

Only about a third of the tweets are offensive -- the classes are **imbalanced**. Compare that to the neatly balanced 20 Newsgroups categories. Imbalance is the norm in real research data (most tweets are not hate speech; most news articles are not about any one topic), and it has serious consequences for how we *evaluate* models. A model that simply says "not offensive" every time is right two-thirds of the time -- yet completely useless. Park that thought; it returns with force on Day 4.

### Comparing the two datasets

How different are tweets from newsgroup posts, as *text*? Let's compare document lengths:

In [ ]:
tw["n_words"] = tw["text"].str.split().str.len()

print(f"20 Newsgroups - median words per document: {df['n_words'].median():.0f}")
print(f"TweetEval     - median words per document: {tw['n_words'].median():.0f}")

Newsgroup posts are typically four to five times longer. This is not a triviality: document length shapes which methods work well. Topic models, for instance, struggle on very short texts (little to go on per document), which is one reason we use the longer newsgroup posts for the unsupervised days and the tweets for the supervised days.

Finally, read some data! Here are a few examples of each class (recall the content note above):

In [ ]:
for label in ["not-offensive", "offensive"]:
    print(f"===== {label} =====")
    for t in tw[tw["label_name"] == label]["text"].sample(3, random_state=7):
        print("-", t)
    print()

Note the `@user` and `http` placeholders: the dataset's creators *anonymised* the tweets by replacing usernames and links. You saw the same idea in the 20 Newsgroups header-stripping -- deciding what to remove from raw text before analysis is a recurring, consequential choice, and we will treat it properly on Day 2.

### Your turn (2)

- **a)** Do offensive tweets differ from non-offensive ones in length? Compute the median `n_words` per class. *Hint: `groupby`.*
- **b)** What fraction of tweets in each class mention `@user`? Any idea what such a difference (or absence of one) could mean?
- **c)** Load a *different* TweetEval task -- for example `"emotion"` -- and find out: how many classes does it have, what are they called, and how balanced are they?

In [ ]:
#your code here


## 3. Your own dataset

If you are bringing your own data to the course (we strongly encourage it!), this is the moment to give it the same treatment. Load it into a DataFrame and answer, at minimum:

1. How many documents does it have?
2. If it has labels or categories: how are they distributed? Balanced or imbalanced?
3. How long are the documents (median and spread)?
4. Read ten random documents. What "mess" do you see -- boilerplate, links, formatting, duplicates?

If your data is a CSV file, `pd.read_csv("path/to/file.csv")` will usually do it (with an Excel file, `pd.read_excel`). If you cannot get your data loaded, don't worry -- bring it along, and we will look at it together during the project sessions.


In [ ]:
#your code here - e.g.:
#mydata = pd.read_csv("path/to/your/data.csv")
#mydata.head()


---

## Solutions to the exercises

No peeking until you have tried!

### Your turn (1)

In [ ]:
#a) longest document and its category
longest = df.loc[df["n_words"].idxmax()]
print(f"longest document: {longest['n_words']} words, category: {longest['category']}")

#b) how many documents mention NASA
mentions_nasa = df["text"].str.contains("NASA")
print(f"documents mentioning NASA: {mentions_nasa.sum()}")

#c) which categories they belong to
print(df[mentions_nasa]["category"].value_counts().head())

### Your turn (2)

In [ ]:
#a) median length per class
print(tw.groupby("label_name")["n_words"].median())

#b) fraction of tweets mentioning @user, per class
tw["has_mention"] = tw["text"].str.contains("@user")
print(tw.groupby("label_name")["has_mention"].mean().round(3))

#c) the emotion task
emotion = load_dataset("cardiffnlp/tweet_eval", "emotion")
print(emotion["train"].features["label"].names)
print(emotion["train"].to_pandas()["label"].value_counts(normalize=True).round(3))

---

**That's it!** You have met both course datasets and refreshed the pandas toolkit we will lean on all week. If anything here felt shaky, the [pandas getting-started guides](https://pandas.pydata.org/docs/getting_started/index.html) and [cssbook.net](https://cssbook.net) (chapters 5-6) are good places to firm up before the course. See you on Day 1!